# City GPT
Character-level Transformer trained on [GeoNames allCountries](https://download.geonames.org/export/dump/) data,
following [Karpathy's GPT](https://github.com/karpathy/ng-video-lecture/blob/master/gpt.py) architecture.

Each training record looks like:
```
🇺🇸🏙️PPL|new york city
🇬🇧⛰️MT|ben nevis
🇯🇵🏙️PPLC|tokyo
```

**Before running:** Settings → Accelerator → GPU T4 x2

In [ ]:
# ── 1. Hyperparameters ─────────────────────────────────────────────────────
import os

BATCH_SIZE    = 64
BLOCK_SIZE    = 256
MAX_ITERS     = 5000
EVAL_INTERVAL = 100
EVAL_ITERS    = 50
EVAL_SAMPLES  = 10
LEARNING_RATE = 3e-4
N_EMBD        = 384
N_HEAD        = 6
N_LAYER       = 6
DROPOUT       = 0.2
VAL_FRACTION  = 0.1
MAX_ROWS      = None   # set e.g. 500_000 for a quick test

DATA_DIR   = "/kaggle/working/data"
ZIP_PATH   = f"{DATA_DIR}/allCountries.zip"
RAW_PATH   = f"{DATA_DIR}/allCountries.txt"
DB_PATH    = f"{DATA_DIR}/corpus.db"
MODEL_PATH = "/kaggle/working/city_gpt.pt"
DATA_URL   = "https://download.geonames.org/export/dump/allCountries.zip"

os.makedirs(DATA_DIR, exist_ok=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
print(f"model checkpoint: {MODEL_PATH}")

In [ ]:
# ── 2. Emoji / record helpers ──────────────────────────────────────────────
_RI_BASE = 0x1F1E6 - ord("A")

def country_to_flag(cc):
    if len(cc) != 2 or not cc.isalpha():
        return "🏳️"
    return "".join(chr(_RI_BASE + ord(c)) for c in cc.upper())

FEATURE_CLASS_EMOJI = {
    "A": "🏛️", "H": "💧", "L": "🌿", "P": "🏙️",
    "R": "🛣️", "S": "🏗️", "T": "⛰️", "U": "🌊", "V": "🌲",
}
_CLASS_FALLBACK = "❓"
FIELD_SEP = "|"
_KEEP_CHARS = frozenset("abcdefghijklmnopqrstuvwxyz0123456789 '-")

def _normalize_name(raw):
    return "".join(ch for ch in raw.lower() if ch in _KEEP_CHARS).strip()

def _record_text(country_code, feature_cls, feature_code, name):
    flag      = country_to_flag(country_code)
    cls_emoji = FEATURE_CLASS_EMOJI.get(feature_cls, _CLASS_FALLBACK)
    return f"{flag}{cls_emoji}{feature_code}{FIELD_SEP}{name}\n"

print("Examples:")
for cc, cls, code, name in [("US","P","PPL","new york city"),("GB","T","MT","ben nevis"),("JP","P","PPLC","tokyo")]:
    print(" ", _record_text(cc, cls, code, name), end="")

In [ ]:
# ── 3. Vocabulary ──────────────────────────────────────────────────────────
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "\n"

def build_vocab():
    """
    id=0  <SOS>  start-of-sequence
    id=1  \\n    end-of-sequence
    id=2+ all other characters sorted
    """
    chars = set()
    for i in range(26):
        chars.add(chr(0x1F1E6 + i))          # regional indicators
    for emoji in FEATURE_CLASS_EMOJI.values():
        chars.update(emoji)
    chars.update(_CLASS_FALLBACK)
    chars.update("🏳️")
    chars.update("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
    chars.add(FIELD_SEP)
    chars.update(_KEEP_CHARS)
    vocab = [SOS_TOKEN, EOS_TOKEN] + sorted(chars)
    stoi  = {ch: i for i, ch in enumerate(vocab)}
    itos  = {i: ch for i, ch in enumerate(vocab)}
    return stoi, itos

stoi, itos = build_vocab()
print(f"Vocab size: {len(stoi)}  (SOS=0, EOS=1)")

In [ ]:
# ── 4. Download + build corpus DB ─────────────────────────────────────────
import sqlite3, urllib.request, zipfile, random
from tqdm import tqdm

COL_NAME=1; COL_ASCIINAME=2; COL_FEATURE_CLASS=6; COL_FEATURE_CODE=7; COL_COUNTRY_CODE=8
_BATCH_INSERT = 10_000
_DB_SCHEMA = """
CREATE TABLE IF NOT EXISTS records (id INTEGER PRIMARY KEY, tokens TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS vocab   (token TEXT PRIMARY KEY, id INTEGER);
"""

def _download(url, dest):
    response = urllib.request.urlopen(url)
    total = int(response.headers.get("Content-Length", 0))
    chunk = 1 << 16
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=os.path.basename(dest)) as bar:
        while True:
            data = response.read(chunk)
            if not data: break
            f.write(data); bar.update(len(data))

if os.path.exists(DB_PATH):
    print(f"corpus.db already exists — skipping prepare.")
    print("Delete it and re-run this cell to rebuild.")
else:
    if not os.path.exists(ZIP_PATH):
        print(f"Downloading {DATA_URL} ...")
        _download(DATA_URL, ZIP_PATH)
    if not os.path.exists(RAW_PATH):
        print("Extracting ...")
        with zipfile.ZipFile(ZIP_PATH) as zf: zf.extractall(DATA_DIR)

    print("Building corpus DB ...")
    con = sqlite3.connect(DB_PATH)
    con.executescript(_DB_SCHEMA)
    con.execute("PRAGMA journal_mode=WAL")
    con.execute("PRAGMA synchronous=NORMAL")

    total_bytes = os.path.getsize(RAW_PATH)
    written = skipped = 0
    buf = []

    with open(RAW_PATH, "r", encoding="utf-8") as fin, \
         tqdm(total=total_bytes, unit="B", unit_scale=True, unit_divisor=1024, desc="corpus") as bar:
        for raw_line in fin:
            bar.update(len(raw_line.encode("utf-8")))
            cols = raw_line.rstrip("\n").split("\t")
            if len(cols) < 19: continue
            country_code = cols[COL_COUNTRY_CODE].strip()
            feature_cls  = cols[COL_FEATURE_CLASS].strip()
            feature_code = cols[COL_FEATURE_CODE].strip()
            if not country_code: continue
            raw_name = cols[COL_ASCIINAME].strip() or cols[COL_NAME].strip()
            name = _normalize_name(raw_name)
            if not name: skipped += 1; continue
            buf.append((_record_text(country_code, feature_cls, feature_code, name),))
            written += 1
            if len(buf) >= _BATCH_INSERT:
                con.executemany("INSERT INTO records (tokens) VALUES (?)", buf)
                con.commit(); buf.clear()
            if MAX_ROWS and written >= MAX_ROWS: break

    if buf:
        con.executemany("INSERT INTO records (tokens) VALUES (?)", buf)
        con.commit()

    con.executemany("INSERT OR REPLACE INTO vocab VALUES (?,?)", stoi.items())
    con.commit(); con.close()
    print(f"Done: {written:,} records, {skipped:,} skipped -> {DB_PATH}")

In [ ]:
# ── 5. Data loading ────────────────────────────────────────────────────────
import torch

def get_batch(id_pool, con, stoi):
    fetch_n = BATCH_SIZE * max(BLOCK_SIZE // 25 + 1, 3)
    chosen  = random.sample(id_pool, min(fetch_n, len(id_pool)))
    rows = con.execute(
        f"SELECT tokens FROM records WHERE id IN ({','.join('?'*len(chosen))})", chosen
    ).fetchall()
    flat = []
    for (text,) in rows:
        flat.extend(stoi[ch] for ch in text if ch in stoi)
    if len(flat) <= BLOCK_SIZE:
        flat = flat * ((BLOCK_SIZE + 2) // max(len(flat), 1) + 1)
    data = torch.tensor(flat, dtype=torch.long)
    ix   = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x    = torch.stack([data[i : i+BLOCK_SIZE]   for i in ix])
    y    = torch.stack([data[i+1 : i+BLOCK_SIZE+1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def estimate_loss(model, con, stoi, train_ids, val_ids):
    model.eval()
    out = {}
    for split, pool in (("train", train_ids), ("val", val_ids)):
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            x, y = get_batch(pool, con, stoi)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

print("Data loading helpers ready.")

In [ ]:
# ── 6. Model (Karpathy GPT) ────────────────────────────────────────────────
import torch.nn as nn
from torch.nn import functional as F

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(N_EMBD, head_size, bias=False)
        self.query = nn.Linear(N_EMBD, head_size, bias=False)
        self.value = nn.Linear(N_EMBD, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.dropout = nn.Dropout(DROPOUT)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T,:T]==0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ self.value(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(head_size * num_heads, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)
    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd), nn.ReLU(),
            nn.Linear(4*n_embd, n_embd), nn.Dropout(DROPOUT),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa   = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.blocks   = nn.Sequential(*[Block(N_EMBD, N_HEAD) for _ in range(N_LAYER)])
        self.ln_f     = nn.LayerNorm(N_EMBD)
        self.lm_head  = nn.Linear(N_EMBD, vocab_size)
        self.apply(self._init_weights)
    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx) + \
            self.position_embedding_table(torch.arange(T, device=DEVICE))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -BLOCK_SIZE:])
            logits = logits[:, -1, :] / temperature
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            idx = torch.cat((idx, torch.multinomial(F.softmax(logits, dim=-1), 1)), dim=1)
        return idx

print("Model defined.")

In [ ]:
# ── 7. Train ───────────────────────────────────────────────────────────────
from tqdm import tqdm, trange

con = sqlite3.connect(DB_PATH, check_same_thread=False)

# Load + split IDs
total   = con.execute("SELECT MAX(id) FROM records").fetchone()[0]
all_ids = list(range(1, total + 1))
random.seed(1337); random.shuffle(all_ids)
n_val     = int(total * VAL_FRACTION)
val_ids   = all_ids[:n_val]
train_ids = all_ids[n_val:]
print(f"Records: {total:,}  (train {len(train_ids):,} / val {len(val_ids):,})  |  vocab: {len(stoi)}  |  device: {DEVICE}")

torch.manual_seed(1337)
model = GPTLanguageModel(len(stoi)).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

sos_id = stoi[SOS_TOKEN]
eos_id = stoi[EOS_TOKEN]
sep_id = stoi[FIELD_SEP]

bar = trange(MAX_ITERS, desc="training", unit="step")
for step in bar:
    if step % EVAL_INTERVAL == 0 or step == MAX_ITERS - 1:
        losses = estimate_loss(model, con, stoi, train_ids, val_ids)
        tqdm.write(f"step {step:5d} / {MAX_ITERS}  |  train loss {losses['train']:.4f}  |  val loss {losses['val']:.4f}")

        # Sample from random val rows
        seed_rows = con.execute(
            f"SELECT tokens FROM records WHERE id IN ({','.join('?'*EVAL_SAMPLES)})",
            random.sample(val_ids, EVAL_SAMPLES),
        ).fetchall()
        samples = []
        for (text,) in seed_rows:
            all_tok = [sos_id] + [stoi[ch] for ch in text if ch in stoi]
            sep_pos = next((i for i, t in enumerate(all_tok) if t == sep_id), None)
            prefix  = all_tok[:sep_pos+1] if sep_pos is not None else all_tok
            out     = model.generate(torch.tensor([prefix], dtype=torch.long, device=DEVICE), max_new_tokens=60)[0].tolist()
            gen     = out[len(prefix):]
            if eos_id in gen: gen = gen[:gen.index(eos_id)]
            samples.append("".join(itos.get(i,"?") for i in prefix[1:]) + "".join(itos.get(i,"?") for i in gen))
        tqdm.write("  samples: " + " | ".join(samples))

        torch.save({"model_state": model.state_dict(), "stoi": stoi, "itos": itos}, MODEL_PATH)
        tqdm.write(f"  checkpoint -> {MODEL_PATH}")

    x, y = get_batch(train_ids, con, stoi)
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    bar.set_postfix(loss=f"{loss.item():.4f}")

con.close()
print(f"Training complete. Model saved to {MODEL_PATH}")

In [ ]:
# ── 8. Generate ────────────────────────────────────────────────────────────
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
stoi  = ckpt["stoi"]; itos = ckpt["itos"]
model = GPTLanguageModel(len(stoi)).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

def generate(prompt="", n=500, temperature=1.0, top_k=None):
    encode = lambda s: [stoi[ch] for ch in s if ch in stoi]
    decode = lambda ids: "".join(itos[i] for i in ids)
    ctx = torch.tensor([[stoi[SOS_TOKEN]] + encode(prompt)], dtype=torch.long, device=DEVICE)
    out = model.generate(ctx, max_new_tokens=n, temperature=temperature, top_k=top_k)
    print(decode(out[0].tolist()))

# Free-form generation
generate(n=500, temperature=1.0)

In [ ]:
# ── 9. Prompted generation ─────────────────────────────────────────────────
# Format: <flag><class-emoji><feature-code>|
# Examples:
#   🇺🇸🏙️PPL|   US populated place
#   🇯🇵⛰️MT|    Japanese mountain
#   🇩🇪💧LK|    German lake
#   🇬🇧🏛️ADM1|  UK administrative region
generate(prompt="🇺🇸🏙️PPL|", n=200, temperature=0.8, top_k=50)